In [16]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path('/data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR')
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

results_dir = project_root / 'results'
results_dir.mkdir(exist_ok=True)

# Experimentation with FlexAnomalies Federated Models

This notebook mirrors the experiment-oriented structure of `experiment_pyod_models.ipynb` and `experiment_transformers_models.ipynb`, but focuses on federated models available in `RADAR.federated_data.algorithms.flexanomalies`.

It evaluates:
- Static-data federated models: `isolationForest`, `pcaAnomaly`, and `clusterAnomaly` on the same UCI datasets used by the PyOD benchmark (`shuttle` and `arrhythmia`).
- Time-series federated models: `autoencoder` and `deepCNN_LSTM` on the same datasets used by the Transformer benchmark (`ai4i_2020_predictive_maintenance_dataset` and `metro_interstate_traffic_volume`).

## Import Required Libraries

Imports for data preparation, federated model training, metrics, and result persistence.

In [17]:
import importlib
import time
from statistics import mean

import numpy as np
import pandas as pd

from flex.pool import FlexPool
from flexanomalies.pool.aggregators_cl import aggregate_cl
from flexanomalies.pool.aggregators_favg import aggregate_ae
from flexanomalies.pool.aggregators_pca import aggregate_pca
from flexanomalies.pool.primitives_cluster import (
    build_server_model_cl,
    copy_model_to_clients_cl,
    get_clients_weights_cl,
    set_aggregated_weights_cl,
    train_cl,
)
from flexanomalies.pool.primitives_deepmodel import (
    build_server_model_ae,
    copy_model_to_clients_ae,
    set_aggregated_weights_ae,
    train_ae,
    weights_collector_ae,
)
from flexanomalies.pool.primitives_iforest import (
    aggregate_if,
    build_server_model_if,
    copy_model_to_clients_if,
    get_clients_weights_if,
    set_aggregated_weights_if,
    train_if,
)
from flexanomalies.pool.primitives_pca import (
    build_server_model_pca,
    copy_model_to_clients_pca,
    get_clients_weights_pca,
    set_aggregated_weights_pca,
    train_pca,
)
from flexanomalies.utils import (
    AutoEncoder,
    ClusterAnomaly,
    DeepCNN_LSTM,
    IsolationForest,
    PCA_Anomaly,
)
from flexanomalies.utils.load_data import federate_data

from RADAR.federated_data.algorithms import flexanomalies
import RADAR.metrics_module as metrics_module
import RADAR.static_data.anomaly_dataset_utils as anomaly_dataset_utils
from RADAR.time_series.preprocessing.preprocessing_ts import StandardScalerPreprocessing
from RADAR.time_series.time_series_datasets_uci import global_load as load_time_series
from RADAR.time_series.time_series_utils import TimeSeriesProcessor

metrics_module = importlib.reload(metrics_module)
anomaly_dataset_utils = importlib.reload(anomaly_dataset_utils)

## Static UCI Benchmark

The static-data benchmark reuses the same anomaly-detection framing as `experiment_pyod_models.ipynb`, so the datasets, train/test philosophy, and contamination logic remain aligned.

In [18]:
static_dataset_configs = {
    'shuttle': anomaly_dataset_utils.build_loaded_uci_anomaly_dataset(
        dataset_name='shuttle',
        normal_label=1,
        target_test_contamination=0.1,
        max_train_normals=8000,
        max_test_size=5000,
    ),
    'arrhythmia': anomaly_dataset_utils.build_loaded_uci_anomaly_dataset(
        dataset_name='arrhythmia',
        normal_label=1,
        target_test_contamination=0.1,
    ),
}

static_summary_df = pd.DataFrame([
    {
        'dataset': dataset_name,
        'samples': config['n_samples'],
        'features': config['n_features'],
        'original_anomaly_ratio': round(config['original_positive_ratio'], 4),
        'benchmark_test_contamination': round(config['benchmark_test_positive_ratio'], 4),
        'train_normals_used': config['train_normals'],
        'test_normals': config['test_normals'],
        'test_anomalies': config['test_anomalies'],
    }
    for dataset_name, config in static_dataset_configs.items()
]).reset_index(drop=True)

display(static_summary_df)

,dataset,samples,features,original_anomaly_ratio,benchmark_test_contamination,train_normals_used,test_normals,test_anomalies
0,shuttle,58000,7,0.214,0.1036,8000,4482,518
1,arrhythmia,452,279,0.458,0.0926,196,49,5


In [19]:
def flatten_1d(values):
    return np.asarray(values).astype(float).ravel()


def binary_1d(values):
    return np.asarray(values).astype(int).ravel()


def compute_metrics_row(y_true, y_pred, scores):
    y_true = binary_1d(y_true)
    y_pred = binary_1d(y_pred)
    scores = flatten_1d(scores)

    limit = min(len(y_true), len(y_pred), len(scores))
    y_true = y_true[:limit]
    y_pred = y_pred[:limit]
    scores = scores[:limit]

    finite_scores = np.isfinite(scores)
    if finite_scores.all() and len(np.unique(y_true)) > 1:
        roc_auc = metrics_module.metric_AUC_ROC_scores(y_true, scores)
        pr_auc = metrics_module.metric_PR_AUC(y_true, scores)
    else:
        roc_auc = np.nan
        pr_auc = np.nan

    return {
        'accuracy': round(metrics_module.metric_accuracy(y_true, y_pred) / 100, 4),
        'precision': round(metrics_module.metric_precision(y_true, y_pred), 4),
        'recall': round(metrics_module.metric_recall(y_true, y_pred), 4),
        'f1': round(metrics_module.metric_F1score(y_true, y_pred), 4),
        'roc_auc_scores': round(float(roc_auc), 4) if np.isfinite(roc_auc) else np.nan,
        'pr_auc_scores': round(float(pr_auc), 4) if np.isfinite(pr_auc) else np.nan,
        'evaluated_samples': int(limit),
    }


direct_flex_model_classes = {
    'isolationForest': IsolationForest,
    'pcaAnomaly': PCA_Anomaly,
    'clusterAnomaly': ClusterAnomaly,
    'autoencoder': AutoEncoder,
    'deepCNN_LSTM': DeepCNN_LSTM,
}


direct_federated_ops = {
    'isolationForest': {
        'build_model': build_server_model_if,
        'copy': copy_model_to_clients_if,
        'train': train_if,
        'collect': get_clients_weights_if,
        'aggregate': aggregate_if,
        'set_weights': set_aggregated_weights_if,
    },
    'pcaAnomaly': {
        'build_model': build_server_model_pca,
        'copy': copy_model_to_clients_pca,
        'train': train_pca,
        'collect': get_clients_weights_pca,
        'aggregate': aggregate_pca,
        'set_weights': set_aggregated_weights_pca,
    },
    'clusterAnomaly': {
        'build_model': build_server_model_cl,
        'copy': copy_model_to_clients_cl,
        'train': train_cl,
        'collect': get_clients_weights_cl,
        'aggregate': aggregate_cl,
        'set_weights': set_aggregated_weights_cl,
    },
    'autoencoder': {
        'build_model': build_server_model_ae,
        'copy': copy_model_to_clients_ae,
        'train': train_ae,
        'collect': weights_collector_ae,
        'aggregate': aggregate_ae,
        'set_weights': set_aggregated_weights_ae,
    },
    'deepCNN_LSTM': {
        'build_model': build_server_model_ae,
        'copy': copy_model_to_clients_ae,
        'train': train_ae,
        'collect': weights_collector_ae,
        'aggregate': aggregate_ae,
        'set_weights': set_aggregated_weights_ae,
    },
}


def extract_prediction_labels(model_object, prediction_output):
    for candidate in (
        getattr(model_object, 'labels_', None),
        getattr(getattr(model_object, 'model', None), 'labels_', None),
        getattr(prediction_output, 'labels_', None),
    ):
        if candidate is not None:
            return binary_1d(candidate)
    return binary_1d(prediction_output)


def predict_and_score_model(model_object, X, y=None):
    prediction_output = model_object.predict(X, y) if y is not None else model_object.predict(X)
    labels = extract_prediction_labels(model_object, prediction_output)
    scores = model_object.decision_function(X, y) if y is not None else model_object.decision_function(X)
    return labels, flatten_1d(scores)


def build_direct_model_kwargs(model_kwargs):
    excluded_keys = {'algorithm_', 'label_parser', 'n_clients', 'n_rounds'}
    return {
        key: value
        for key, value in model_kwargs.items()
        if key not in excluded_keys
    }


def train_platform_model(model_kwargs, X_train, y_train):
    model = flexanomalies.FlexAnomalyDetection(**model_kwargs)
    model.fit(X_train, y_train)
    return model


def train_direct_federated_model(model_kwargs, X_train, y_train):
    algorithm_name = model_kwargs['algorithm_']
    direct_model_cls = direct_flex_model_classes[algorithm_name]
    direct_model = direct_model_cls(**build_direct_model_kwargs(model_kwargs))
    federated_ops = direct_federated_ops[algorithm_name]

    flex_dataset = federate_data(model_kwargs['n_clients'], X_train, y_train)
    pool = FlexPool.client_server_pool(
        fed_dataset=flex_dataset,
        server_id=f'{algorithm_name}_server',
        init_func=federated_ops['build_model'],
        model=direct_model,
    )

    for _ in range(model_kwargs['n_rounds']):
        pool.servers.map(federated_ops['copy'], pool.clients)
        pool.clients.map(federated_ops['train'])
        pool.aggregators.map(federated_ops['collect'], pool.clients)
        if algorithm_name == 'clusterAnomaly':
            pool.aggregators.map(federated_ops['aggregate'], model=direct_model)
        else:
            pool.aggregators.map(federated_ops['aggregate'])
        pool.aggregators.map(federated_ops['set_weights'], pool.servers)

    return pool.servers._models[f'{algorithm_name}_server']['model']


def benchmark_average_time(train_callable, repetitions):
    execution_times = []
    trained_model = None
    for _ in range(repetitions):
        start_time = time.perf_counter()
        trained_model = train_callable()
        execution_times.append(time.perf_counter() - start_time)
    return mean(execution_times), trained_model


static_model_configs = [
    {
        'algorithm_': 'isolationForest',
        'n_estimators': 100,
        'n_clients': 5,
        'n_rounds': 5,
    },
    {
        'algorithm_': 'pcaAnomaly',
        'preprocess': False,
        'n_components': 4,
        'n_clients': 5,
        'n_rounds': 5,
    },
    {
        'algorithm_': 'clusterAnomaly',
        'n_clusters': 4,
        'n_clients': 5,
        'n_rounds': 5,
    },
]

static_results = []

for dataset_name, config in static_dataset_configs.items():
    print(f'\nStatic dataset: {dataset_name}')
    y_train_dummy = np.zeros(config['X_train'].shape[0], dtype=int)

    for model_params in static_model_configs:
        model_kwargs = {
            **model_params,
            'contamination': float(config['benchmark_test_positive_ratio']),
            'label_parser': None,
        }

        model = train_platform_model(model_kwargs, config['X_train'], y_train_dummy)
        predictions, scores = predict_and_score_model(model, config['X_test'])

        metrics_row = compute_metrics_row(config['y_test'], predictions, scores)
        result_row = {
            'data_type': 'static',
            'dataset': dataset_name,
            'algorithm': model_params['algorithm_'],
            'contamination': round(float(config['benchmark_test_positive_ratio']), 4),
            **metrics_row,
        }
        static_results.append(result_row)
        print(result_row)

static_results_df = pd.DataFrame(static_results).sort_values(
    ['dataset', 'pr_auc_scores', 'roc_auc_scores'],
    ascending=[True, False, False],
    na_position='last'
).reset_index(drop=True)

display(static_results_df)


Static dataset: shuttle
Federated Params:{'n_clients': 5, 'n_rounds': 5} 
 Model Params:{'algorithm_': 'isolationForest', 'n_estimators': 100, 'contamination': 0.1036, 'label_parser': None}

Running round: 0

Training model at client.
Training model at client.
Training model at client.
Training model at client.
Training model at client.

Running round: 1

Training model at client.
Training model at client.
Training model at client.
Training model at client.
Training model at client.

Running round: 2

Training model at client.
Training model at client.
Training model at client.
Training model at client.
Training model at client.

Running round: 3

Training model at client.
Training model at client.
Training model at client.
Training model at client.
Training model at client.

Running round: 4

Training model at client.
Training model at client.
Training model at client.
Training model at client.
Training model at client.
Inspecting model's attributes:
contamination: 0.1036
n_estimator

/data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR/.venv/lib/python3.10/site-packages/sklearn/ensemble/_iforest.py:336: UserWarning: max_samples (1000) is greater than the total number of samples (39). max_samples will be set to n_samples for estimation.
  warn(


Training model at client.


/data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR/.venv/lib/python3.10/site-packages/sklearn/ensemble/_iforest.py:336: UserWarning: max_samples (1000) is greater than the total number of samples (39). max_samples will be set to n_samples for estimation.
  warn(


Training model at client.


...

/data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR/.venv/lib/python3.10/site-packages/sklearn/ensemble/_iforest.py:336: UserWarning: max_samples (1000) is greater than the total number of samples (39). max_samples will be set to n_samples for estimation.
  warn(


Training model at client.


/data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR/.venv/lib/python3.10/site-packages/sklearn/ensemble/_iforest.py:336: UserWarning: max_samples (1000) is greater than the total number of samples (39). max_samples will be set to n_samples for estimation.
  warn(


Inspecting model's attributes:
contamination: 0.09259259259259259
n_estimators: 100
max_samples: 1000
max_features: 1.0
bootstrap: False
n_jobs: 1
behaviour: old
random_state: None
verbose: 0
model_path: 
model: IsolationForest(contamination=0.09259259259259259, max_samples=1000, n_jobs=1)
algorithm_: isolationForest
label_parser: None
{'data_type': 'static', 'dataset': 'arrhythmia', 'algorithm': 'isolationForest', 'contamination': 0.0926, 'accuracy': np.float64(0.8889), 'precision': np.float64(0.4), 'recall': np.float64(0.4), 'f1': np.float64(0.4), 'roc_auc_scores': 0.7224, 'pr_auc_scores': 0.2461, 'evaluated_samples': 54}
Federated Params:{'n_clients': 5, 'n_rounds': 5} 
 Model Params:{'algorithm_': 'pcaAnomaly', 'preprocess': False, 'n_components': 4, 'contamination': 0.09259259259259259, 'label_parser': None}

Running round: 0

Training model at client.
Training model at client.
Training model at client.
Training model at client.
Training model at client.

Running round: 1

Trainin

,data_type,dataset,algorithm,contamination,accuracy,precision,recall,f1,roc_auc_scores,pr_auc_scores,evaluated_samples
0,static,arrhythmia,pcaAnomaly,0.0926,0.8889,0.4000,0.4000,0.4000,0.7673,0.5309,54
1,static,arrhythmia,clusterAnomaly,0.0926,0.8889,0.4000,0.4000,0.4000,0.7673,0.5309,54
2,static,arrhythmia,isolationForest,0.0926,0.8889,0.4000,0.4000,0.4000,0.7224,0.2461,54
3,static,shuttle,clusterAnomaly,0.1036,0.9356,0.6892,0.6892,0.6892,0.9682,0.7919,5000
4,static,shuttle,isolationForest,0.1036,0.9184,0.6062,0.6062,0.6062,0.9114,0.6768,5000
5,static,shuttle,pcaAnomaly,0.1036,0.9084,0.5579,0.5579,0.5579,0.7974,0.5860,5000


## Time-Series UCI Benchmark

The time-series benchmark follows the same datasets and preprocessing logic already used in `experiment_transformers_models.ipynb`, but swaps the modeling layer for federated FlexAnomalies models.

- `autoencoder` is tested in reconstruction mode.
- `deepCNN_LSTM` is tested in forecasting mode with a one-step horizon.

In [20]:
WINDOW_SIZE = 24
STEP_SIZE = 1
TEST_SIZE = 0.2
METRO_LOW_Q = 0.05
METRO_HIGH_Q = 0.95

def chronological_split(X, y, test_size=0.2):
    split_idx = int(len(X) * (1 - test_size))
    return X[:split_idx], X[split_idx:], y[:split_idx], y[split_idx:]

def aggregate_window_labels(y_windows):
    y_windows = np.asarray(y_windows)
    if y_windows.ndim == 1:
        return y_windows.astype(int)
    return (y_windows.reshape(y_windows.shape[0], -1).sum(axis=1) > 0).astype(int)

def prepare_ai4i_dataset(test_size=TEST_SIZE):
    X, y = load_time_series('ai4i_2020_predictive_maintenance_dataset')
    labels = y['Machine failure'].astype(int).to_numpy()
    X = X.drop(columns=['Type'], errors='ignore')

    scaler = StandardScalerPreprocessing()
    X_scaled = scaler.fit_transform(X)
    X_values = np.asarray(X_scaled, dtype=np.float32)

    X_train, X_test, y_train, y_test = chronological_split(X_values, labels, test_size=test_size)
    return {
        'dataset': 'ai4i_2020_predictive_maintenance_dataset',
        'X_train': np.asarray(X_train, dtype=np.float32),
        'X_test': np.asarray(X_test, dtype=np.float32),
        'y_train': np.asarray(y_train, dtype=int),
        'y_test': np.asarray(y_test, dtype=int),
        'n_samples': len(X_values),
        'n_features': X_values.shape[1],
        'positive_ratio_points': round(float(np.mean(labels)), 4),
        'label_note': 'Machine failure from UCI target',
    }

def prepare_metro_dataset(test_size=TEST_SIZE, low_q=METRO_LOW_Q, high_q=METRO_HIGH_Q):
    X, y = load_time_series('metro_interstate_traffic_volume')
    traffic_volume = y['traffic_volume'].astype(float)
    low_threshold = float(traffic_volume.quantile(low_q))
    high_threshold = float(traffic_volume.quantile(high_q))
    labels = ((traffic_volume <= low_threshold) | (traffic_volume >= high_threshold)).astype(int).to_numpy()

    X = X.drop(columns=['date_time', 'holiday', 'weather_main', 'weather_description'], errors='ignore')

    scaler = StandardScalerPreprocessing()
    X_scaled = scaler.fit_transform(X)
    X_values = np.asarray(X_scaled, dtype=np.float32)

    X_train, X_test, y_train, y_test = chronological_split(X_values, labels, test_size=test_size)
    return {
        'dataset': 'metro_interstate_traffic_volume',
        'X_train': np.asarray(X_train, dtype=np.float32),
        'X_test': np.asarray(X_test, dtype=np.float32),
        'y_train': np.asarray(y_train, dtype=int),
        'y_test': np.asarray(y_test, dtype=int),
        'n_samples': len(X_values),
        'n_features': X_values.shape[1],
        'positive_ratio_points': round(float(np.mean(labels)), 4),
        'label_note': f'Extreme traffic volume: <= q{low_q:.2f} or >= q{high_q:.2f}',
        'low_threshold': round(low_threshold, 3),
        'high_threshold': round(high_threshold, 3),
    }

time_series_dataset_configs = {
    'ai4i': prepare_ai4i_dataset(),
    'metro_interstate': prepare_metro_dataset(),
}

time_series_summary_df = pd.DataFrame([
    {
        'dataset_key': dataset_key,
        'dataset_name': config['dataset'],
        'samples': config['n_samples'],
        'features': config['n_features'],
        'positive_ratio_points': config['positive_ratio_points'],
        'label_note': config['label_note'],
    }
    for dataset_key, config in time_series_dataset_configs.items()
]).reset_index(drop=True)

display(time_series_summary_df)

Metadata: {'uci_id': 601, 'name': 'AI4I 2020 Predictive Maintenance Dataset', 'repository_url': 'https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset', 'data_url': 'https://archive.ics.uci.edu/static/public/601/data.csv', 'abstract': 'The AI4I 2020 Predictive Maintenance Dataset is a synthetic dataset that reflects real predictive maintenance data encountered in industry.', 'area': 'Computer Science', 'tasks': ['Classification', 'Regression', 'Causal-Discovery'], 'characteristics': ['Multivariate', 'Time-Series'], 'num_instances': 10000, 'num_features': 6, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'], 'index_col': ['UID', 'Product ID'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2020, 'last_updated': 'Wed Feb 14 2024', 'dataset_doi': '10.24432/C5HS5C', 'creators': [], 'intro_paper': {'ID': 386, 'type': 'NATIVE', 'title': 'Explainable Artificial 

,dataset_key,dataset_name,samples,features,positive_ratio_points,label_note
0,ai4i,ai4i_2020_predictive_maintenance_dataset,10000,5,0.0339,Machine failure from UCI target
1,metro_interstate,metro_interstate_traffic_volume,48204,4,0.1006,Extreme traffic volume: <= q0.05 or >= q0.95


In [21]:
def align_binary_targets(targets, expected_len):
    targets = np.asarray(targets)
    window_targets = aggregate_window_labels(targets)
    flat_targets = binary_1d(targets)

    if len(window_targets) == expected_len:
        return window_targets
    if len(flat_targets) == expected_len:
        return flat_targets
    if len(window_targets) > expected_len:
        return window_targets[:expected_len]
    if len(flat_targets) > expected_len:
        return flat_targets[:expected_len]
    return np.resize(window_targets, expected_len).astype(int)


def align_scores(scores, expected_len):
    scores = flatten_1d(scores)
    if len(scores) >= expected_len:
        return scores[:expected_len]
    return np.resize(scores, expected_len)


def resolve_contamination_ratio(ratio, default=0.1, min_value=0.01, max_value=0.5):
    if ratio is None or not np.isfinite(ratio):
        return float(default)
    return float(np.clip(ratio, min_value, max_value - 1e-6))


def build_autoencoder_windows(config, window_size=WINDOW_SIZE, step_size=STEP_SIZE):
    processor = TimeSeriesProcessor(window_size=window_size, step_size=step_size, future_prediction=False)
    X_train_windows, y_train_windows, X_test_windows, y_test_windows = processor.process_train_test(
        config['X_train'], config['y_train'], config['X_test'], config['y_test']
    )
    return X_train_windows, y_train_windows, X_test_windows, y_test_windows, aggregate_window_labels(y_test_windows)


def build_forecasting_windows(config, window_size=WINDOW_SIZE, step_size=STEP_SIZE, n_pred=1):
    processor = TimeSeriesProcessor(window_size=window_size, step_size=step_size, future_prediction=True, n_pred=n_pred)
    X_train_windows, y_train_windows, X_test_windows, y_test_windows, label_test_windows = processor.process_train_test(
        config['X_train'], config['y_train'], config['X_test'], config['y_test'], l_test=config['y_test']
    )
    return X_train_windows, y_train_windows, X_test_windows, y_test_windows, aggregate_window_labels(label_test_windows)


time_series_model_configs = [
    {
        'algorithm_': 'autoencoder',
        'builder': build_autoencoder_windows,
        'base_kwargs': {
            'epochs': 5,
            'batch_size': 16,
            'neurons': [32, 16, 32],
            'hidden_act': ['relu', 'relu', 'relu'],
            'preprocess': False,
            'w_size': WINDOW_SIZE,
            'n_pred': 1,
            'n_clients': 3,
            'n_rounds': 5,
        },
    },
    {
        'algorithm_': 'deepCNN_LSTM',
        'builder': build_forecasting_windows,
        'base_kwargs': {
            'epochs': 5,
            'batch_size': 8,
            'filters_cnn': [8, 6],
            'units_lstm': [8, 6],
            'kernel_size': [4, 4],
            'hidden_act': ['relu', 'relu'],
            'w_size': WINDOW_SIZE,
            'n_pred': 1,
            'n_clients': 3,
            'n_rounds': 3,
        },
    },
]

time_series_results = []

for dataset_key, config in time_series_dataset_configs.items():
    print(f'\nTime-series dataset: {config["dataset"]}')

    for model_config in time_series_model_configs:
        X_train_windows, y_train_windows, X_test_windows, y_test_windows, y_eval_reference = model_config['builder'](config)
        contamination = resolve_contamination_ratio(config.get('positive_ratio_points'))

        model_kwargs = {
            'algorithm_': model_config['algorithm_'],
            'contamination': contamination,
            'label_parser': None,
            'input_dim': int(config['n_features']),
            **model_config['base_kwargs'],
        }

        model = train_platform_model(model_kwargs, X_train_windows, y_train_windows)

        if model_config['algorithm_'] == 'deepCNN_LSTM':
            predictions, raw_scores = predict_and_score_model(model, X_test_windows, y_test_windows)
        else:
            predictions, raw_scores = predict_and_score_model(model, X_test_windows)

        scores = align_scores(raw_scores, len(predictions))
        y_true = align_binary_targets(y_eval_reference, len(predictions))
        metrics_row = compute_metrics_row(y_true, predictions, scores)

        result_row = {
            'data_type': 'time_series',
            'dataset': dataset_key,
            'dataset_name': config['dataset'],
            'algorithm': model_config['algorithm_'],
            'window_size': WINDOW_SIZE,
            'contamination': round(contamination, 4),
            'train_windows': int(len(X_train_windows)),
            'test_windows': int(len(X_test_windows)),
            **metrics_row,
        }
        time_series_results.append(result_row)
        print(result_row)

time_series_results_df = pd.DataFrame(time_series_results).sort_values(
    ['dataset', 'pr_auc_scores', 'roc_auc_scores'],
    ascending=[True, False, False],
    na_position='last'
).reset_index(drop=True)

display(time_series_results_df)


Time-series dataset: ai4i_2020_predictive_maintenance_dataset
Federated Params:{'n_clients': 3, 'n_rounds': 5} 
 Model Params:{'algorithm_': 'autoencoder', 'contamination': 0.0339, 'label_parser': None, 'input_dim': 5, 'epochs': 5, 'batch_size': 16, 'neurons': [32, 16, 32], 'hidden_act': ['relu', 'relu', 'relu'], 'preprocess': False, 'w_size': 24, 'n_pred': 1}


/data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR/.venv/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Running round: 0



/data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR/.venv/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 18 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Training model at client.
Epoch 1/5
120/120 ━━━━━━━━━━━━━━━━━━━━ 54s 456ms/step - loss: 1.23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.9396   ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.802 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.704 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.641 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.6152 - val_loss: 0.0717
Epoch 2/5
120/120 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 0.06 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0628 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.057 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.053 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0506 - val_loss: 0.0176
Epoch 3/5
120/120 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 0.01 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0168 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.015 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.014 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.013 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.012 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0128 - val_loss: 0.0051
Epoch 4

...

 Non-trainable params: 0 (0.00 B)

None

Running round: 0



/data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR/.venv/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 26 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Training model at client.
Epoch 1/5
1156/1156 ━━━━━━━━━━━━━━━━━━━━ 18:06 941ms/step - loss: 0.46 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.4514     ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.450 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.446 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.445 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.443 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.441 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.437 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.430 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.422 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.414 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.406 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.400 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.393 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.387 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.382 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.381 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.378 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.373 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.369 ━━━━━

,data_type,dataset,dataset_name,algorithm,window_size,contamination,train_windows,test_windows,accuracy,precision,recall,f1,roc_auc_scores,pr_auc_scores,evaluated_samples
0,time_series,ai4i,ai4i_2020_predictive_maintenance_dataset,autoencoder,24,0.0339,7977,1977,0.6753,0.4000,0.0031,0.0062,0.5147,0.3440,1977
1,time_series,ai4i,ai4i_2020_predictive_maintenance_dataset,deepCNN_LSTM,24,0.0339,7976,1976,0.6174,0.0456,0.9231,0.0870,0.8576,0.1216,1976
2,time_series,metro_interstate,metro_interstate_traffic_volume,autoencoder,24,0.1006,38540,9618,0.2518,1.0000,0.0006,0.0011,0.4835,0.7441,9618
3,time_series,metro_interstate,metro_interstate_traffic_volume,deepCNN_LSTM,24,0.1006,38539,9617,0.6140,0.0942,0.3614,0.1494,0.5058,0.0975,9617


In [22]:
static_results_path = results_dir / 'uci_flexanomalies_static_results.csv'
time_series_results_path = results_dir / 'uci_flexanomalies_time_series_results.csv'
summary_results_path = results_dir / 'uci_flexanomalies_summary.csv'

static_results_df.to_csv(static_results_path, index=False)
time_series_results_df.to_csv(time_series_results_path, index=False)

summary_df = pd.concat([
    static_results_df,
    time_series_results_df,
], ignore_index=True, sort=False)
summary_df.to_csv(summary_results_path, index=False)

print(f'Saved static results to: {static_results_path}')
print(f'Saved time-series results to: {time_series_results_path}')
print(f'Saved combined summary to: {summary_results_path}')
display(summary_df)

Saved static results to: /data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR/results/uci_flexanomalies_static_results.csv
Saved time-series results to: /data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR/results/uci_flexanomalies_time_series_results.csv
Saved combined summary to: /data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR/results/uci_flexanomalies_summary.csv


,data_type,dataset,algorithm,contamination,accuracy,precision,recall,f1,roc_auc_scores,pr_auc_scores,evaluated_samples,dataset_name,window_size,train_windows,test_windows
0,static,arrhythmia,pcaAnomaly,0.0926,0.8889,0.4000,0.4000,0.4000,0.7673,0.5309,54,NaN,NaN,NaN,NaN
1,static,arrhythmia,clusterAnomaly,0.0926,0.8889,0.4000,0.4000,0.4000,0.7673,0.5309,54,NaN,NaN,NaN,NaN
2,static,arrhythmia,isolationForest,0.0926,0.8889,0.4000,0.4000,0.4000,0.7224,0.2461,54,NaN,NaN,NaN,NaN
3,static,shuttle,clusterAnomaly,0.1036,0.9356,0.6892,0.6892,0.6892,0.9682,0.7919,5000,NaN,NaN,NaN,NaN
4,static,shuttle,isolationForest,0.1036,0.9184,0.6062,0.6062,0.6062,0.9114,0.6768,5000,NaN,NaN,NaN,NaN
5,static,shuttle,pcaAnomaly,0.1036,0.9084,0.5579,0.5579,0.5579,0.7974,0.5860,5000,NaN,NaN,NaN,NaN
6,time_series,ai4i,autoencoder,0.0339,0.6753,0.4000,0.0031,0.0062,0.5147,0.3440,1977,ai4i_2020_predictive_maintenance_dataset,24.0,7977.0,1977.0
7,time_series,ai4i,deepCNN_LSTM,0.0339,0.6174,0.0456,0.9231,0.0870,0.8576,0.1216,1976,ai4i_2020_predictive_maintenance_dataset,24.0,7976.0,1976.0
8,time_series,metro_interstate,autoencoder,0.1006,0.2518,1.0000,0.0006,0.0011,0.4835,0.7441,9618,metro_interstate_traffic_volume,24.0,38540.0,9618.0
9,time_series,metro_interstate,deepCNN_LSTM,0.1006,0.6140,0.0942,0.3614,0.1494,0.5058,0.0975,9617,metro_interstate_traffic_volume,24.0,38539.0,9617.0


## Timing Comparison: RADAR Platform vs Direct FlexAnomalies Library

This section mirrors the timing idea used in `experiment_pyod_models.ipynb`, but here the direct baseline uses the native `flexanomalies` primitives and pool API instead of the RADAR wrapper.

- `platform_time_s`: average training time using `RADAR.federated_data.algorithms.flexanomalies.FlexAnomalyDetection`.
- `base_time_s`: average training time using the direct `flexanomalies` library flow.
- `overhead_s`: difference `platform_time_s - base_time_s`.
- `speedup_base_over_platform`: ratio `base_time_s / platform_time_s`.

For practicality, the benchmark uses few repetitions because federated deep-learning models are much heavier than the local PyOD estimators.

In [ ]:
TIMING_REPETITIONS = {
    'static': 3,
    'time_series': 1,
}


timing_results = []

for dataset_name, config in static_dataset_configs.items():
    y_train_dummy = np.zeros(config['X_train'].shape[0], dtype=int)

    for model_params in static_model_configs:
        model_kwargs = {
            **model_params,
            'contamination': float(config['benchmark_test_positive_ratio']),
            'label_parser': None,
        }

        platform_time_s, platform_model = benchmark_average_time(
            lambda mk=model_kwargs, ds=config, yt=y_train_dummy: train_platform_model(mk, ds['X_train'], yt),
            repetitions=TIMING_REPETITIONS['static'],
        )
        base_time_s, base_model = benchmark_average_time(
            lambda mk=model_kwargs, ds=config, yt=y_train_dummy: train_direct_federated_model(mk, ds['X_train'], yt),
            repetitions=TIMING_REPETITIONS['static'],
        )

        platform_predictions, platform_scores = predict_and_score_model(platform_model, config['X_test'])
        base_predictions, base_scores = predict_and_score_model(base_model, config['X_test'])

        platform_metrics = compute_metrics_row(config['y_test'], platform_predictions, platform_scores)
        base_metrics = compute_metrics_row(config['y_test'], base_predictions, base_scores)

        timing_results.append({
            'data_type': 'static',
            'dataset': dataset_name,
            'algorithm': model_params['algorithm_'],
            'timing_repetitions': TIMING_REPETITIONS['static'],
            'platform_time_s': round(platform_time_s, 4),
            'base_time_s': round(base_time_s, 4),
            'overhead_s': round(platform_time_s - base_time_s, 4),
            'speedup_base_over_platform': round(base_time_s / platform_time_s, 4) if platform_time_s > 0 else np.nan,
            'platform_roc_auc_scores': platform_metrics['roc_auc_scores'],
            'base_roc_auc_scores': base_metrics['roc_auc_scores'],
            'roc_auc_diff': round(platform_metrics['roc_auc_scores'] - base_metrics['roc_auc_scores'], 4)
            if np.isfinite(platform_metrics['roc_auc_scores']) and np.isfinite(base_metrics['roc_auc_scores'])
            else np.nan,
            'platform_pr_auc_scores': platform_metrics['pr_auc_scores'],
            'base_pr_auc_scores': base_metrics['pr_auc_scores'],
            'pr_auc_diff': round(platform_metrics['pr_auc_scores'] - base_metrics['pr_auc_scores'], 4)
            if np.isfinite(platform_metrics['pr_auc_scores']) and np.isfinite(base_metrics['pr_auc_scores'])
            else np.nan,
        })

for dataset_key, config in time_series_dataset_configs.items():
    for model_config in time_series_model_configs:
        X_train_windows, y_train_windows, X_test_windows, y_test_windows, y_eval_reference = model_config['builder'](config)
        contamination = resolve_contamination_ratio(config.get('positive_ratio_points'))

        model_kwargs = {
            'algorithm_': model_config['algorithm_'],
            'contamination': contamination,
            'label_parser': None,
            'input_dim': int(config['n_features']),
            **model_config['base_kwargs'],
        }

        platform_time_s, platform_model = benchmark_average_time(
            lambda mk=model_kwargs, Xw=X_train_windows, yw=y_train_windows: train_platform_model(mk, Xw, yw),
            repetitions=TIMING_REPETITIONS['time_series'],
        )
        base_time_s, base_model = benchmark_average_time(
            lambda mk=model_kwargs, Xw=X_train_windows, yw=y_train_windows: train_direct_federated_model(mk, Xw, yw),
            repetitions=TIMING_REPETITIONS['time_series'],
        )

        if model_config['algorithm_'] == 'deepCNN_LSTM':
            platform_predictions, platform_raw_scores = predict_and_score_model(platform_model, X_test_windows, y_test_windows)
            base_predictions, base_raw_scores = predict_and_score_model(base_model, X_test_windows, y_test_windows)
        else:
            platform_predictions, platform_raw_scores = predict_and_score_model(platform_model, X_test_windows)
            base_predictions, base_raw_scores = predict_and_score_model(base_model, X_test_windows)

        platform_scores = align_scores(platform_raw_scores, len(platform_predictions))
        base_scores = align_scores(base_raw_scores, len(base_predictions))
        y_true_platform = align_binary_targets(y_eval_reference, len(platform_predictions))
        y_true_base = align_binary_targets(y_eval_reference, len(base_predictions))

        platform_metrics = compute_metrics_row(y_true_platform, platform_predictions, platform_scores)
        base_metrics = compute_metrics_row(y_true_base, base_predictions, base_scores)

        timing_results.append({
            'data_type': 'time_series',
            'dataset': dataset_key,
            'dataset_name': config['dataset'],
            'algorithm': model_config['algorithm_'],
            'timing_repetitions': TIMING_REPETITIONS['time_series'],
            'platform_time_s': round(platform_time_s, 4),
            'base_time_s': round(base_time_s, 4),
            'overhead_s': round(platform_time_s - base_time_s, 4),
            'speedup_base_over_platform': round(base_time_s / platform_time_s, 4) if platform_time_s > 0 else np.nan,
            'platform_roc_auc_scores': platform_metrics['roc_auc_scores'],
            'base_roc_auc_scores': base_metrics['roc_auc_scores'],
            'roc_auc_diff': round(platform_metrics['roc_auc_scores'] - base_metrics['roc_auc_scores'], 4)
            if np.isfinite(platform_metrics['roc_auc_scores']) and np.isfinite(base_metrics['roc_auc_scores'])
            else np.nan,
            'platform_pr_auc_scores': platform_metrics['pr_auc_scores'],
            'base_pr_auc_scores': base_metrics['pr_auc_scores'],
            'pr_auc_diff': round(platform_metrics['pr_auc_scores'] - base_metrics['pr_auc_scores'], 4)
            if np.isfinite(platform_metrics['pr_auc_scores']) and np.isfinite(base_metrics['pr_auc_scores'])
            else np.nan,
        })

timing_results_df = pd.DataFrame(timing_results).sort_values(
    ['data_type', 'dataset', 'speedup_base_over_platform'],
    ascending=[True, True, False],
    na_position='last',
).reset_index(drop=True)

display(timing_results_df)

In [24]:
timing_results_path = results_dir / 'uci_flexanomalies_timing_results.csv'
timing_results_df.to_csv(timing_results_path, index=False)

print(f'Saved timing results to: {timing_results_path}')
display(timing_results_df)

Saved timing results to: /data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR/results/uci_flexanomalies_timing_results.csv


,data_type,dataset,algorithm,timing_repetitions,platform_time_s,base_time_s,overhead_s,speedup_base_over_platform,platform_roc_auc_scores,base_roc_auc_scores,roc_auc_diff,platform_pr_auc_scores,base_pr_auc_scores,pr_auc_diff,dataset_name
0,static,arrhythmia,isolationForest,3,3.4179,3.8487,-0.4308,1.1260,0.7429,0.6571,0.0858,0.2590,0.2348,0.0242,NaN
1,static,arrhythmia,clusterAnomaly,3,0.1365,0.0975,0.0390,0.7140,0.7429,0.7429,0.0000,0.5207,0.5207,0.0000,NaN
2,static,arrhythmia,pcaAnomaly,3,0.8352,0.4615,0.3737,0.5525,0.7673,0.7673,0.0000,0.5309,0.5309,0.0000,NaN
3,static,shuttle,clusterAnomaly,3,0.1718,0.1659,0.0059,0.9657,0.9719,0.9681,0.0038,0.8251,0.7926,0.0325,NaN
4,static,shuttle,pcaAnomaly,3,0.0971,0.0929,0.0042,0.9567,0.7974,0.7974,0.0000,0.5860,0.5860,0.0000,NaN
5,static,shuttle,isolationForest,3,3.9139,3.7301,0.1839,0.9530,0.9208,0.9195,0.0013,0.6817,0.6764,0.0053,NaN
6,time_series,ai4i,deepCNN_LSTM,1,58.0987,58.9432,-0.8445,1.0145,0.8528,0.8867,-0.0339,0.1180,0.1417,-0.0237,ai4i_2020_predictive_maintenance_dataset
7,time_series,ai4i,autoencoder,1,35.0265,33.7133,1.3132,0.9625,0.4634,0.5062,-0.0428,0.3197,0.3496,-0.0299,ai4i_2020_predictive_maintenance_dataset
8,time_series,metro_interstate,autoencoder,1,74.1797,78.4012,-4.2215,1.0569,0.4425,0.4649,-0.0224,0.7126,0.7262,-0.0136,metro_interstate_traffic_volume
9,time_series,metro_interstate,deepCNN_LSTM,1,154.8378,146.9297,7.9081,0.9489,0.5194,0.5145,0.0049,0.1025,0.1008,0.0017,metro_interstate_traffic_volume


## Notes

- The static section intentionally uses the same anomaly-benchmark preparation as the PyOD experiment to keep the comparison fair.
- The time-series section intentionally uses the same raw datasets and preprocessing ideas as the Transformer experiment.
- Federated models are more expensive than lightweight local baselines, so the number of rounds and epochs is kept moderate for practical execution.